# Análise de Dados e Viabilidade de Predição de Alagamentos – Sorocaba

**Projeto Integrador – Univesp**

Este notebook utiliza os dados brutos disponibilizados para avaliar se é possível construir um modelo de previsão de alagamento e quais limitações existem.

## Arquivos utilizados
- `areas_risco_defesa_civil_sorocaba.csv` – áreas de risco monitoradas
- `eventos_alagamento_sorocaba_documentados.csv` – eventos de alagamento documentados
- `sibh_data_eawz4ux.csv` – dados de precipitação (SIBH / posto Jardim Paes de Almeida)


## 1. Carregamento e exploração inicial

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

BASE = Path('/home/workdir/attachments')

areas = pd.read_csv(BASE / 'areas_risco_defesa_civil_sorocaba.csv')
eventos = pd.read_csv(BASE / 'eventos_alagamento_sorocaba_documentados.csv')
chuva = pd.read_csv(BASE / 'sibh_data_eawz4ux.csv')

print('=== Áreas de risco ===')
print(areas.shape)
display(areas.head(8))
print('\n=== Eventos documentados ===')
print(eventos.shape)
display(eventos)
print('\n=== Dados de chuva (SIBH) ===')
print(chuva.shape)
print(chuva['Nome'].unique())
display(chuva.head(3))


## 2. Preparação dos dados de chuva

In [ ]:
# Converter data e limpar
chuva['Data'] = pd.to_datetime(chuva['Data'], format='%Y/%m/%d %H:%M', errors='coerce')
chuva = chuva.dropna(subset=['Data'])
chuva['Valor'] = pd.to_numeric(chuva['Valor'], errors='coerce').fillna(0)

print('Período disponível:', chuva['Data'].min(), '→', chuva['Data'].max())
print('Total de registros:', len(chuva))
print('Registros com chuva > 0:', (chuva['Valor'] > 0).sum())
print('\nEstatísticas de precipitação (mm):')
print(chuva['Valor'].describe())

# Agregar por dia
chuva_diaria = chuva.set_index('Data').resample('D')['Valor'].sum().rename('chuva_mm')
print('\nDias com maior acumulado de chuva:')
print(chuva_diaria.sort_values(ascending=False).head(12))


## 3. Análise dos eventos de alagamento documentados

In [ ]:
eventos['data_evento'] = pd.to_datetime(eventos['data_evento'], errors='coerce')
display(eventos[['data_evento', 'chuva_mm', 'nivel_rio_sorocaba_m', 'vias_pontos_criticos_atingidos']])

print('\n--- Observações ---')
print('Total de eventos documentados:', len(eventos))
print('Eventos com valor de chuva preenchido:', eventos['chuva_mm'].notna().sum())
print('Eventos com nível do rio preenchido:', eventos['nivel_rio_sorocaba_m'].notna().sum())


## 4. Tentativa de cruzamento chuva × eventos

In [ ]:
# Verificar chuva nos dias dos eventos (e dias anteriores)
for _, row in eventos.iterrows():
    d = row['data_evento']
    if pd.isna(d):
        continue
    janela = chuva_diaria.loc[d - pd.Timedelta(days=2) : d]
    print(f'\nEvento {d.date()} | chuva reportada: {row["chuva_mm"]} | nível rio: {row["nivel_rio_sorocaba_m"]}')
    if len(janela):
        print(janela.to_string())
    else:
        print('  (sem dados de chuva na janela)')


## 5. Avaliação da viabilidade de um modelo preditivo

### O que temos
- Lista de áreas de risco (útil para contextualizar, mas não é série temporal).
- Apenas **5 eventos** de alagamento documentados (rótulos positivos muito escassos).
- Série de precipitação de **um único posto** (Jardim Paes de Almeida), com resolução irregular.
- Quase **nenhuma medição consistente de nível do Rio Sorocaba** alinhada aos eventos.
- Os valores de chuva reportados nos eventos (ex.: 113,8 mm em 2026-03-07) **não batem** com o acumulado do posto SIBH no mesmo dia.

### Conclusão técnica
**Com os dados brutos atuais NÃO é viável treinar um modelo supervisionado confiável de previsão de alagamento.**

Motivos principais:
1. Pouquíssimos exemplos positivos (eventos) → risco alto de overfitting ou modelo inútil.
2. Falta de variável-alvo bem definida e alinhada temporalmente (alagou? sim/não em cada intervalo).
3. Ausência de série de nível do rio (feature crítica para alagamento fluvial).
4. Chuva de apenas um posto, que pode não representar os pontos críticos (Dom Aguirre, Parque das Águas etc.).
5. Inconsistência entre chuva reportada nos eventos e a série SIBH.

### Recomendações – mudança na forma de implantação dos dados


In [ ]:
print('''
RECOMENDAÇÕES PRÁTICAS:

1. Buscar série de NÍVEL DO RIO SOROCABA
   - Postos fluviométricos DAEE: 4E-018 (Sorocaba), 4E-019 (Iperó)
   - HidroWeb (ANA) / SIBH – baixar nível (cota) além da chuva

2. Ampliar a base de eventos rotulados
   - Coletar mais datas de interdição da Semob / Defesa Civil / notícias
   - Criar rótulo diário: 1 = houve alagamento/interdição naquele dia, 0 = não

3. Usar mais de um pluviômetro
   - Postos próximos aos pontos críticos (Éden, Iperó, etc.)

4. Abordagem alternativa (mais realista no curto prazo):
   - Regras / limiares baseados em conhecimento de domínio
     (ex.: se chuva_24h > 50 mm OU nível_rio > limiar → alerta)
   - Ou detecção de anomalia / modelo semi-supervisionado
   - Manter o dashboard + alertas Telegram com regras simples + visualização

5. Dados de exemplo / sintéticos
   - Continuar usando o CSV de exemplo gerado anteriormente para demonstrar
     o fluxo de ML, deixando claro que é protótipo acadêmico.
''')


## 6. Protótipo com regras simples (demonstração)

Mesmo sem modelo treinado robusto, podemos implementar uma regra de alerta baseada em limiares, útil para o dashboard e Telegram.


In [ ]:
def classificar_risco(chuva_24h, nivel_rio=None):
    '''Regra simples de risco (protótipo).
    nivel_rio em metros (opcional). Retorna 0=Baixo, 1=Médio, 2=Alto.
    '''
    if nivel_rio is not None and nivel_rio > 3.4:
        return 2
    if chuva_24h >= 60:
        return 2
    if chuva_24h >= 30 or (nivel_rio is not None and nivel_rio > 2.7):
        return 1
    return 0

# Exemplo com os maiores dias de chuva da série
maiores = chuva_diaria.sort_values(ascending=False).head(10)
print('Classificação por regra simples (apenas chuva diária):')
for data, mm in maiores.items():
    risco = classificar_risco(mm)
    label = {0: 'Baixo', 1: 'Médio', 2: 'Alto'}[risco]
    print(f'  {data.date()} | {mm:5.1f} mm → risco {label}')


## 7. Resumo final

| Aspecto | Situação atual | Ação recomendada |
|---------|----------------|------------------|
| Eventos rotulados | 5 apenas | Ampliar com mais fontes (Semob, Defesa Civil, imprensa) |
| Chuva | 1 posto, série longa | Manter + buscar postos próximos aos pontos críticos |
| Nível do rio | Quase inexistente nos arquivos | **Prioridade**: baixar cota dos postos fluviométricos DAEE/ANA |
| Modelo ML supervisionado | Não viável ainda | Usar regras + protótipo com dados de exemplo |
| Produto do PI | Viável | Dashboard + alertas Telegram com regras; documentar limitações |

O projeto acadêmico continua válido como protótipo de sistema de apoio à decisão. A predição estatística robusta exige dados adicionais (principalmente nível do rio e mais eventos rotulados).
